In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import accuracy_score, f1_score, classification_report, ConfusionMatrixDisplay

RESULTS_FOLDER = Path("../../results/LR_group_results")
RESULTS_FOLDER.mkdir(parents=True, exist_ok=True)

ACTIVITIES = ["walking", "walking_upstairs", "walking_downstairs"]
LABEL_COLUMNS = ["split", "activity", "participant", "position", "platform", "recording_id"]

# Load the features made by build_features.py
data = pd.read_csv("../../data/processed/group/group_combined.csv")
X = data.drop(columns=LABEL_COLUMNS)
y = data["activity"]

print(f"{len(data):,} windows x {X.shape[1]} features")
print(data.groupby(["participant", "activity"]).size().unstack(fill_value=0))

1,634,462 windows x 7 features
activity     walking  walking_downstairs  walking_upstairs
participant                                               
P01           628632               47795             57012
P02           411260               42190             50276
P03           296437               42244             58616


In [ ]:
def run_logistic_regression(is_test, name, file_name):
    # Scale, train, test, print the results and returns the macro F1.
    X_train, X_test = X[~is_test], X[is_test]
    y_train, y_test = y[~is_test], y[is_test]

    # Learn the scaling from the training data only, then apply it to both
    scaler = MinMaxScaler(feature_range=(-1, 1))
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)

    # Train and predict (class_weight handles the walking-heavy imbalance)
    lr = LogisticRegression(max_iter=2000, class_weight="balanced")
    lr.fit(X_train_scaled, y_train)
    predictions = lr.predict(X_test_scaled)

    # Results
    macro_f1 = f1_score(y_test, predictions, average="macro")
    print(f"===== Logistic Regression: {name} =====")
    print(f"Train: {len(y_train):,} windows   Test: {len(y_test):,} windows")
    print(f"Accuracy: {accuracy_score(y_test, predictions):.3f}")
    print(f"Macro F1: {macro_f1:.3f}\n")
    print(classification_report(y_test, predictions, labels=ACTIVITIES, digits=3, zero_division=0))

    # Confusion matrix
    ConfusionMatrixDisplay.from_predictions(y_test, predictions, labels=ACTIVITIES, normalize="true",
                                            cmap="Blues", xticks_rotation=30)
    plt.title(f"Logistic Regression: {name}")
    plt.tight_layout()
    plt.savefig(RESULTS_FOLDER / f"LR_group_{file_name}_confusion_matrix.png", dpi=150)
    plt.show()

    # Scores per activity graph
    report = pd.DataFrame(classification_report(y_test, predictions, labels=ACTIVITIES,
                                                output_dict=True, zero_division=0)).T
    per_activity = report.loc[ACTIVITIES, ["precision", "recall", "f1-score"]]
    per_activity.plot(kind="bar", figsize=(7, 4), ylim=(0, 1), rot=0)
    plt.title(f"Logistic Regression: scores per activity ({name})")
    plt.ylabel("Score")
    plt.legend(loc="lower right")
    plt.tight_layout()
    plt.savefig(RESULTS_FOLDER / f"LR_group_{file_name}_per_activity.png", dpi=150)
    plt.show()

    return macro_f1

In [ ]:
# 70/30 split by recording (read from the split column)
results = {}

is_test = data["split"] == "test"
results["70/30 by recording"] = run_logistic_regression(is_test, "70/30 by recording", "70_30")

In [ ]:
# Train on two people, test on the third (3 runs, alternating test person)
for person in sorted(data["participant"].unique()):
    is_test = data["participant"] == person
    position = data.loc[is_test, "position"].iloc[0]
    name = f"train on others, test on {person} ({position})"
    results[f"Test on {person} ({position})"] = run_logistic_regression(is_test, name, f"test_{person}")

In [ ]:
# Summary of the 4 runs results
held_out_scores = [score for split, score in results.items() if split.startswith("Test on")]
results["Leave-one-participant-out average"] = sum(held_out_scores) / len(held_out_scores)

summary = pd.Series(results, name="Macro F1").round(3).to_frame()
summary.to_csv(RESULTS_FOLDER / "LR_group_summary.csv")
summary